# exp_004 — Agent context growth

This notebook analyzes measured agent trajectories only. Fixture smoke output is harness validation and is rejected before plotting. The recheck protocol uses trajectory length 1 as a zero-distractor one-turn control, followed by 4/8/16/32 observations.

All conclusions are descriptive and condition-limited.

In [ ]:
import csv
import json
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from llm_lab.analysis import (
    aggregate_agent_trials,
    plot_reliability_by_length,
    plot_reliability_by_position,
    require_measured_trials,
    validate_complete_matrix,
)
from llm_lab.evaluation import load_trial_results

RESULTS = ROOT / 'experiments/exp_004-agent_context_growth/results'
RUN_LABEL = os.environ.get('EXP004_RUN_LABEL', 'main')
MANIFEST_PATH = RESULTS / 'manifests' / f'{RUN_LABEL}.json'
RAW_PATH = RESULTS / 'raw' / ('trials.jsonl' if RUN_LABEL == 'main' else f'{RUN_LABEL}-trials.jsonl')
PROCESSED_PATH = RESULTS / 'processed' / ('summary.csv' if RUN_LABEL == 'main' else f'{RUN_LABEL}-summary.csv')
FIGURE_SUFFIX = '' if RUN_LABEL == 'main' else f'-{RUN_LABEL}'
for required_path in (MANIFEST_PATH, RAW_PATH, PROCESSED_PATH):
    if not required_path.is_file():
        raise FileNotFoundError(f'measured exp_004 input is required: {required_path}')

run_manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
if run_manifest.get('fixture_only'):
    raise ValueError('fixture_only exp_004 results cannot be reported as Qwen measurements')
source_manifest = Path(run_manifest['source_manifest']['path'])
if not source_manifest.is_absolute():
    source_manifest = ROOT / source_manifest
if not source_manifest.is_file():
    raise FileNotFoundError(f'resolved source_manifest is required: {source_manifest}')

with PROCESSED_PATH.open(newline='', encoding='utf-8') as source:
    processed_rows = list(csv.DictReader(source))
if not processed_rows:
    raise ValueError('processed exp_004 summary.csv must contain measured rows')

measured_trials = require_measured_trials(load_trial_results(RAW_PATH))
rows = aggregate_agent_trials(measured_trials)
protocol = run_manifest['protocol']
output_policy = protocol.get('output_policy')
retry_policy = protocol.get('retry_policy')
one_turn_control = protocol.get('one_turn_control')
if one_turn_control is None or one_turn_control.get('trajectory_length') != 1:
    raise ValueError('exp_004 manifest must declare the one-turn control')
if (
    not isinstance(output_policy, dict)
    or output_policy.get('format') != 'single_json_object'
    or output_policy.get('markdown_allowed') is not False
    or output_policy.get('max_new_tokens') != protocol.get('sampling', {}).get('max_new_tokens')
):
    raise ValueError('exp_004 output policy must require one JSON object without markdown')
if not isinstance(retry_policy, dict) or retry_policy.get('max_attempts') != 3 or retry_policy.get('backoff_seconds') != 0.0:
    raise ValueError('exp_004 retry policy must be fixed at three attempts with no backoff')
variant_ids = [item['condition_id'] for item in run_manifest['source_manifest']['variants']]
task_types = protocol['task_types']
validate_complete_matrix(
    rows,
    variant_ids=variant_ids,
    trajectory_lengths=protocol['trajectory_lengths'],
    critical_positions=protocol['critical_positions'],
    task_types=task_types,
)

# Rows retain final_task_success, critical_fact_reuse_rate, and trajectory_context_tokens.
failure_category_counts = {
    (row['variant_condition_id'], row['trajectory_length'], row['requested_critical_position']): row['failure_category_counts']
    for row in rows
}
plot_reliability_by_length(rows, RESULTS / f'figures/reliability-vs-trajectory-length{FIGURE_SUFFIX}.png')
plot_reliability_by_position(rows, RESULTS / f'figures/reliability-vs-critical-position{FIGURE_SUFFIX}.png')
rows


## Interpretation checklist

Inspect final task success and critical-fact reuse against trajectory/context length and critical-information position. Report tool-call validity, repeated actions, recoveries, total input tokens, and `failure_category_counts` separately. Distinguish retrieval/state-tracking from tool/planning failures only when the logged trajectory makes the distinction observable. Do not call fixture results model findings.